In [1]:
print("allok")

allok


In [1]:
import os
import getpass
import pandas as pd

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")


In [4]:
RAG_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL ="text-embedding-3-small"
DEEPEVAL_JUDGE_MODEL = "gpt-4.1-mini"

In [5]:
os.environ[
    "DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"
] = "300"

os.environ[
    "DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"
] = "600"

os.environ[
    "DEEPEVAL_RETRY_MAX_ATTEMPTS"
] = "1"

In [6]:
documents = [
    Document(
        page_content="Full-time employees receive 24 paid leaves per calendar year.",
        metadata={"doc_id": "leave_policy"},
    ),
    Document(
        page_content="Employees are allowed to work from home for a maximum of 2 days per week.",
        metadata={"doc_id": "remote_policy"},
    ),
    Document(
        page_content="Employees can claim up to ₹3000 per month for internet reimbursement.",
        metadata={"doc_id": "internet_policy"},
    ),
    Document(
        page_content="The standard probation period for new employees is 6 months.",
        metadata={"doc_id": "probation_policy"},
    ),
    Document(
        page_content="Employees receive ₹1000 per month as mobile reimbursement.",
        metadata={"doc_id": "mobile_policy"},
    ),
    Document(
        page_content="Medical insurance coverage begins from the employee's date of joining.",
        metadata={"doc_id": "insurance_policy"},
    ),
]


In [7]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [8]:
vector_store = InMemoryVectorStore(embedding=embeddings)

In [9]:
vector_store.add_documents(documents)

['1a963067-e3dc-489f-9034-82db2c0f79d2',
 'c18e93b3-d86e-4ddc-b274-c294566a711d',
 'fffec764-0ff8-4c62-ba42-28c347a245d4',
 '92abd987-396b-4a25-8166-62faa512cdbe',
 '47968daf-893f-4bfe-a914-efab501c77fa',
 '8c3a0a5e-d6ac-4250-a276-25e6c06d50ed']

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [11]:
llm = ChatOpenAI(
    model=RAG_MODEL,
    temperature=0,
)

In [12]:
def rag_pipeline(query: str) -> dict:
    retrieved_docs = retriever.invoke(query)

    retrieval_context = [
        doc.page_content
        for doc in retrieved_docs
    ]

    retrieved_doc_ids = [
        doc.metadata.get("doc_id")
        for doc in retrieved_docs
    ]

    context = "\n\n".join(retrieval_context)

    prompt = f"""
You are an HR policy assistant.

Answer the user's question ONLY from the supplied context.

Rules:
1. Do not use outside knowledge.
2. Do not invent policy details.
3. If the answer is not present in the context, say:
   "I don't know based on the provided context."
4. Keep the answer concise.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "retrieval_context": retrieval_context,
        "retrieved_doc_ids": retrieved_doc_ids,
    }


In [13]:
sample = rag_pipeline("What is the monthly internet reimbursement limit?")

print("ANSWER:")
print(sample["answer"])

print("\nRETRIEVED DOCS:")
print(sample["retrieved_doc_ids"])

print("\nCONTEXT:")
for chunk in sample["retrieval_context"]:
    print("-", chunk)

ANSWER:
The monthly internet reimbursement limit is ₹3000.

RETRIEVED DOCS:
['internet_policy', 'mobile_policy', 'remote_policy']

CONTEXT:
- Employees can claim up to ₹3000 per month for internet reimbursement.
- Employees receive ₹1000 per month as mobile reimbursement.
- Employees are allowed to work from home for a maximum of 2 days per week.


In [14]:
goldens = [
    Golden(
        input="How many paid leaves does a full-time employee receive?",
        expected_output="A full-time employee receives 24 paid leaves per calendar year.",
    ),
    Golden(
        input="How many work-from-home days are allowed per week?",
        expected_output="Employees can work from home for a maximum of 2 days per week.",
    ),
    Golden(
        input="What is the monthly internet reimbursement limit?",
        expected_output="Employees can claim up to ₹3000 per month for internet reimbursement.",
    ),
    Golden(
        input="What is the probation period for new employees?",
        expected_output="The standard probation period for new employees is 6 months.",
    ),
    Golden(
        input="Can an employee work remotely for 3 days every week?",
        expected_output="No. Employees can work from home for a maximum of 2 days per week.",
    ),
    Golden(
        input="When does employee medical insurance coverage begin?",
        expected_output="Medical insurance coverage begins from the employee's date of joining.",
    ),
]

dataset = EvaluationDataset(goldens=goldens)

print("Goldens:", len(dataset.goldens))


Goldens: 6


In [15]:
rag_runs = []

In [16]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(id=None, input='How many paid leaves does a full-time employee receive?', actual_output=None, expected_output='A full-time employee receives 24 paid leaves per calendar year.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(id=None, input='How many work-from-home days are allowed per week?', actual_output=None, expected_output='Employees can work from home for a maximum of 2 days per week.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(id=None, input='What is the monthly internet reimbursement limit?', actual_output=None, expected_output='Employees can claim up to ₹3000 per month for 

In [17]:
dataset.goldens

[Golden(id=None, input='How many paid leaves does a full-time employee receive?', actual_output=None, expected_output='A full-time employee receives 24 paid leaves per calendar year.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None),
 Golden(id=None, input='How many work-from-home days are allowed per week?', actual_output=None, expected_output='Employees can work from home for a maximum of 2 days per week.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None),
 Golden(id=None, input='What is the monthly internet reimbursement limit?', actual_output=None, expected_output='Employees can claim up to ₹3000 per month for internet reimbursement.', context=None,

In [18]:
for i, golden in enumerate(dataset.goldens, start=1):
    print(f"Running RAG {i}/{len(dataset.goldens)}")
    result = rag_pipeline(golden.input)
    rag_runs.append({
        "input": golden.input,
        "expected_output": golden.expected_output,
        "actual_output": result["answer"],
        "retrieval_context": result["retrieval_context"],
        "retrieved_doc_ids": result["retrieved_doc_ids"],
    })


Running RAG 1/6
Running RAG 2/6
Running RAG 3/6
Running RAG 4/6
Running RAG 5/6
Running RAG 6/6


In [19]:
pd.DataFrame([
    {
        "input": run["input"],
        "expected_output": run["expected_output"],
        "actual_output": run["actual_output"],
        "retrieved_doc_ids": run["retrieved_doc_ids"],
    }
    for run in rag_runs
])

,input,expected_output,actual_output,retrieved_doc_ids
0,How many paid leaves does a full-time employee...,A full-time employee receives 24 paid leaves p...,A full-time employee receives 24 paid leaves p...,"[leave_policy, remote_policy, mobile_policy]"
1,How many work-from-home days are allowed per w...,Employees can work from home for a maximum of ...,Employees are allowed to work from home for a ...,"[remote_policy, leave_policy, internet_policy]"
2,What is the monthly internet reimbursement limit?,Employees can claim up to ₹3000 per month for ...,The monthly internet reimbursement limit is ₹3...,"[internet_policy, mobile_policy, remote_policy]"
3,What is the probation period for new employees?,The standard probation period for new employee...,The probation period for new employees is 6 mo...,"[probation_policy, insurance_policy, remote_po..."
4,Can an employee work remotely for 3 days every...,No. Employees can work from home for a maximum...,"No, employees are allowed to work from home fo...","[remote_policy, internet_policy, leave_policy]"
5,When does employee medical insurance coverage ...,Medical insurance coverage begins from the emp...,Employee medical insurance coverage begins fro...,"[insurance_policy, leave_policy, internet_policy]"
